# cAST-Scope — run réel sur GPU (StarCoder2-7B / CodeLlama-7B)

Compare les 3 baselines de chunking (`fixed`, `cast_orig`, `cast_scope`) sur RepoEval et CrossCodeEval, avec le même retriever BM25 et le même générateur pour les 3 — voir `Robertkiza0/cAST-state` (https://github.com/Robertkiza0/cAST-state) pour le code et les tests locaux (`--generator stub`, déjà validés sans GPU).

## 0. Vérifier le GPU

In [ ]:
!nvidia-smi

## 1. Cloner le dépôt cAST-state + installer les dépendances

In [ ]:
!git clone --depth 1 https://github.com/Robertkiza0/cAST-state.git
%cd cAST-state
!pip install -q -r requirements.txt
!pip install -q transformers accelerate editdistance


## 1bis. (Optionnel) Token Hugging Face

In [ ]:
from google.colab import userdata
import os

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN chargé.")
except Exception:
    print("Pas de secret HF_TOKEN configuré — pas grave, pas nécessaire pour un modèle public.")


## 2. Télécharger les données RepoEval (tâches + 8 dépôts réels)

Mêmes chemins que `datasets_io.py` attend par défaut (`datasets rapo/` et `data/repos_source/`) — aucune option à passer à `run_benchmark.py` ensuite.

In [ ]:
!git clone --no-checkout --depth 1 https://github.com/microsoft/CodeT.git codet_src
%cd codet_src
!git sparse-checkout init --cone
!git sparse-checkout set RepoCoder
!git checkout main
%cd ..

import zipfile

with zipfile.ZipFile('codet_src/RepoCoder/datasets/datasets.zip') as z:
    z.extractall('datasets rapo')
with zipfile.ZipFile('codet_src/RepoCoder/repositories/line_and_api_level.zip') as z:
    z.extractall('data/repos_source')

print('RepoEval : dataset et dépôts extraits.')


## 3. Télécharger les données CrossCodeEval (tâches Python + carte des licences)

Les vrais dépôts GitHub seront clonés à la volée par `run_benchmark.py` lui-même (voir `datasets_io.ensure_repo_cloned`), dans `cceval_repos/` — rien à faire ici à part récupérer les tâches.

In [ ]:
!git clone --no-checkout --depth 1 https://github.com/amazon-science/cceval.git cceval_src
%cd cceval_src
!git sparse-checkout init --cone
!git sparse-checkout set data
!git checkout main
%cd ..

import tarfile

with tarfile.open('cceval_src/data/crosscodeeval_data.tar.xz') as tar:
    tar.extractall('crosscodeeval_data')

print('CrossCodeEval : tâches extraites.')


## 4. Test rapide, générateur factice (valide tout le pipeline avant le vrai run GPU)

In [ ]:
!python run_benchmark.py --dataset repoeval --n-tasks 10 --generator stub


## 5. Run réel — StarCoder2-7B

`--n-tasks 300` correspond à l'échelle déjà utilisée pour les runs McNemar de ce projet (voir `repocoder-mine/colab_weighted_ast_retrieval.ipynb`). Réduire `--n-tasks` pour un essai plus rapide avant de lancer le run complet.

In [ ]:
!python run_benchmark.py --dataset both --n-tasks 300 --generator hf \
    --model-name bigcode/starcoder2-7b --device cuda


## 6. (Optionnel) Run réel — CodeLlama-7B-Python

Même protocole, second générateur, pour vérifier que le résultat chunking ne dépend pas du choix de modèle (cf. spec: "Conserve le MÊME générateur/LLM ... pour les 3 baselines" — ceci compare plutôt across-run si le classement des 3 baselines est stable d'un générateur à l'autre, une vérification de robustesse utile pour le papier).

In [ ]:
!python run_benchmark.py --dataset both --n-tasks 300 --generator hf \
    --model-name codellama/CodeLlama-7b-Python-hf --device cuda


## Notes

- Pass@1 == Exact Match ici (pas de harnais d'exécution sur ces variantes line-level de RepoEval/CrossCodeEval — voir `metrics.py:compute_pass_at_1`).
- Le clonage à la volée des dépôts CrossCodeEval peut échouer pour certaines tâches (dépôt supprimé/privé depuis) — `run_benchmark.py` les ignore et continue, en l'indiquant dans la sortie.
- Pour relancer seulement le tableau final sans re-télécharger les données, ré-exécuter uniquement les cellules 5/6.